In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
DATA_PATH = Path(r"C:\Users\Rahul\Desktop\Codes\aiml\Machine Learning\Classification\phishingEmail\data\raw\meajor_cleaned_preprocessed.parquet.gzip")

print(DATA_PATH)
print("Exists:", DATA_PATH.exists())

C:\Users\Rahul\Desktop\Codes\aiml\Machine Learning\Classification\phishingEmail\data\raw\meajor_cleaned_preprocessed.parquet.gzip
Exists: True


In [3]:
df = pd.read_parquet(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (108685, 20)


In [4]:
print("Columns:")
for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

print(df.dtypes)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

df.head()

missing = df.isna().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print("Columns containing missing values:")
print(missing)

print(df.columns.tolist())


print(df["label"].value_counts(dropna=False))
print()
print(df["label"].value_counts(normalize=True, dropna=False))

Columns:
1. sender
2. sender_domain
3. receiver
4. receiver_domain
5. date
6. subject
7. content_types
8. body
9. urls
10. url_count
11. url_length_max
12. url_length_avg
13. url_subdom_max
14. url_subdom_avg
15. attachment_count
16. has_attachments
17. attachment_types
18. language
19. source
20. label
sender               object
sender_domain        object
receiver             object
receiver_domain      object
date                 object
subject              object
content_types        object
body                 object
urls                 object
url_count           float64
url_length_max      float64
url_length_avg      float64
url_subdom_max      float64
url_subdom_avg      float64
attachment_count    float64
has_attachments        bool
attachment_types     object
language             object
source               object
label               float64
dtype: object
Columns containing missing values:
attachment_types    107019
urls                 44633
receiver_domain       2371
subje

In [5]:
print("Label counts:")
print(df["label"].value_counts(dropna=False))

print("\nLabel percentages:")
print(
    df["label"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)


print("Number of unique sources:")
print(df["source"].nunique())

print("\nSource distribution:")
print(df["source"].value_counts(dropna=False))


source_label = pd.crosstab(
    df["source"],
    df["label"],
    normalize="index"
) * 100

print(source_label.round(2))


print("Missing values:")
print(
    df[["subject", "body", "urls"]]
    .isna()
    .sum()
)

print("\nEmpty strings:")
for col in ["subject", "body", "urls"]:
    empty = (df[col].fillna("").astype(str).str.strip() == "").sum()
    print(f"{col}: {empty}")



print("Exact duplicate rows:", df.duplicated().sum())


print("Duplicate bodies:", df["body"].duplicated().sum())


print(
    "Duplicate subject + body:",
    df.duplicated(subset=["subject", "body"]).sum()
)



label_conflicts = (
    df.groupby(["subject", "body"])["label"]
      .nunique()
)

conflicts = label_conflicts[label_conflicts > 1]

print("Subject + body combinations with conflicting labels:", len(conflicts))


Label counts:
label
0.0    60650
1.0    48034
NaN        1
Name: count, dtype: int64

Label percentages:
label
0.0    55.8
1.0    44.2
NaN     0.0
Name: proportion, dtype: float64
Number of unique sources:
3

Source distribution:
source
trec5    49583
trec7    44096
trec6    15005
None         1
Name: count, dtype: int64
label     0.0    1.0
source              
trec5   60.36  39.64
trec6   74.70  25.30
trec7   44.25  55.75
Missing values:
subject     1455
body           1
urls       44633
dtype: int64

Empty strings:
subject: 1455
body: 1
urls: 44633
Exact duplicate rows: 13
Duplicate bodies: 5166
Duplicate subject + body: 3751
Subject + body combinations with conflicting labels: 0


In [6]:
source_label = pd.crosstab(
    df["source"],
    df["label"],
    normalize="index"
) * 100

print(source_label.round(2))


df["body_length"] = df["body"].fillna("").astype(str).str.len()
df["subject_length"] = df["subject"].fillna("").astype(str).str.len()

print("Body length:")
print(df["body_length"].describe())

print("\nSubject length:")
print(df["subject_length"].describe())

print("URL count:")
print(df["url_count"].describe())

print("\nEmails with at least one URL:")
print((df["url_count"] > 0).sum())

print("\nPercentage with at least one URL:")
print(round((df["url_count"] > 0).mean() * 100, 2), "%")

label     0.0    1.0
source              
trec5   60.36  39.64
trec6   74.70  25.30
trec7   44.25  55.75
Body length:
count    108685.000000
mean       1168.102857
std        1141.702402
min           0.000000
25%         380.000000
50%         797.000000
75%        1533.000000
max       30574.000000
Name: body_length, dtype: float64

Subject length:
count    108685.000000
mean         42.040870
std          47.495656
min           0.000000
25%          20.000000
50%          32.000000
75%          47.000000
max        3701.000000
Name: subject_length, dtype: float64
URL count:
count    108685.000000
mean          3.308138
std          10.507161
min           0.000000
25%           0.000000
50%           1.000000
75%           2.000000
max         553.000000
Name: url_count, dtype: float64

Emails with at least one URL:
64052

Percentage with at least one URL:
58.93 %


In [7]:
df_clean = df.copy()

print("Original shape:", df_clean.shape)


df_clean = df_clean.dropna(subset=["label"]).copy()

print("After removing missing labels:", df_clean.shape)


df_clean["label"] = df_clean["label"].astype(int)

print(df_clean["label"].value_counts())


before = len(df_clean)

df_clean = df_clean.drop_duplicates().copy()

after = len(df_clean)

print("Rows removed:", before - after)
print("Current shape:", df_clean.shape)



df_clean["subject"] = df_clean["subject"].fillna("").astype(str)
df_clean["body"] = df_clean["body"].fillna("").astype(str)

df_clean["text"] = (
    df_clean["subject"].str.strip()
    + " "
    + df_clean["body"].str.strip()
).str.strip()




before = len(df_clean)

df_clean = df_clean[df_clean["text"].str.len() > 0].copy()

after = len(df_clean)

print("Empty emails removed:", before - after)
print("Current shape:", df_clean.shape)


duplicate_texts = df_clean["text"].duplicated().sum()

print("Duplicate subject + body texts:", duplicate_texts)


df_clean[["subject", "body", "text", "label"]].head(5)


Original shape: (108685, 22)
After removing missing labels: (108684, 22)
label
0    60650
1    48034
Name: count, dtype: int64
Rows removed: 13
Current shape: (108671, 22)
Empty emails removed: 0
Current shape: (108671, 23)
Duplicate subject + body texts: 3745


,subject,body,text,label
0,[ORGANIZATION] failover plan.,"Hi [NAME], \n\nTonight we are rolling out a new report. Currently, only you and [NAME] have ac...","[ORGANIZATION] failover plan. Hi [NAME], \n\nTonight we are rolling out a new report. Currentl...",0
1,RE: Intranet Site,"[NAME] r these new?\tIntranet Site\n\n[NAME],\nWe need a few links to added to the [ORGANIZATION...","RE: Intranet Site [NAME] r these new?\tIntranet Site\n\n[NAME],\nWe need a few links to added to...",0
2,FW: [ORGANIZATION] Company information,"[NAME]/[NAME],\n\nWe are currently trading under GTC Spot contracts with [ORGANIZATION] for [ORG...","FW: [ORGANIZATION] Company information [NAME]/[NAME],\n\nWe are currently trading under GTC Spot...",0
3,New Master Physical,[NAME] and [NAME] -\n\nAttached is a worksheet for a new master physical with [ORGANIZATION]. P...,New Master Physical [NAME] and [NAME] -\n\nAttached is a worksheet for a new master physical wit...,0
4,FW: [ORGANIZATION]/Mirant GISB,FYI. Below is a copy of my communication with [ORGANIZATION] regarding our GISB in case someone ...,FW: [ORGANIZATION]/Mirant GISB FYI. Below is a copy of my communication with [ORGANIZATION] rega...,0


In [8]:
import hashlib

df_clean["email_group"] = (
    df_clean["subject"].str.strip()
    + "\n"
    + df_clean["body"].str.strip()
).map(
    lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest()
)

print("Rows:", len(df_clean))
print("Unique email groups:", df_clean["email_group"].nunique())



group_sizes = df_clean["email_group"].value_counts()

print("Groups containing duplicates:", (group_sizes > 1).sum())
print("Largest group size:", group_sizes.max())
print("Total rows belonging to duplicate groups:", group_sizes[group_sizes > 1].sum())

Rows: 108671
Unique email groups: 104926
Groups containing duplicates: 888
Largest group size: 667
Total rows belonging to duplicate groups: 4633


In [9]:
print("Total rows:", len(df_clean))
print("Unique email groups:", df_clean["email_group"].nunique())


group_df = (
    df_clean[["email_group", "label"]]
    .drop_duplicates("email_group")
    .reset_index(drop=True)
)

print("Number of unique groups:", len(group_df))
print("\nGroup-level label distribution:")
print(group_df["label"].value_counts())

Total rows: 108671
Unique email groups: 104926
Number of unique groups: 104926

Group-level label distribution:
label
0    58084
1    46842
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_group_idx, temp_group_idx = next(
    splitter.split(
        group_df["email_group"],
        group_df["label"]
    )
)

train_groups = set(
    group_df.iloc[train_group_idx]["email_group"]
)

temp_groups = set(
    group_df.iloc[temp_group_idx]["email_group"]
)

print("Training groups:", len(train_groups))
print("Temporary groups:", len(temp_groups))

Training groups: 83940
Temporary groups: 20986


In [11]:
temp_df = group_df[
    group_df["email_group"].isin(temp_groups)
].reset_index(drop=True)

splitter_test = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    splitter_test.split(
        temp_df["email_group"],
        temp_df["label"]
    )
)

val_groups = set(
    temp_df.iloc[val_idx]["email_group"]
)

test_groups = set(
    temp_df.iloc[test_idx]["email_group"]
)

print("Validation groups:", len(val_groups))
print("Test groups:", len(test_groups))

Validation groups: 10493
Test groups: 10493


In [12]:
train_df = df_clean[
    df_clean["email_group"].isin(train_groups)
].copy()

val_df = df_clean[
    df_clean["email_group"].isin(val_groups)
].copy()

test_df = df_clean[
    df_clean["email_group"].isin(test_groups)
].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


train_group_set = set(train_df["email_group"])
val_group_set = set(val_df["email_group"])
test_group_set = set(test_df["email_group"])

print("Train ∩ Validation:", len(train_group_set & val_group_set))
print("Train ∩ Test:", len(train_group_set & test_group_set))
print("Validation ∩ Test:", len(val_group_set & test_group_set))

Train: (86933, 24)
Validation: (10976, 24)
Test: (10762, 24)
Train: (86933, 24)
Validation: (10976, 24)
Test: (10762, 24)
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [13]:
def show_distribution(name, data):
    print(f"\n{name}")
    print("-" * len(name))
    print(data["label"].value_counts())
    print(
        data["label"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

show_distribution("TRAIN", train_df)
show_distribution("VALIDATION", val_df)
show_distribution("TEST", test_df)


for data in [train_df, val_df, test_df]:
    data.drop(
        columns=["body_length", "subject_length", "email_group"],
        errors="ignore",
        inplace=True
    )


TRAIN
-----
label
0    48568
1    38365
Name: count, dtype: int64
label
0    55.87
1    44.13
Name: proportion, dtype: float64

VALIDATION
----------
label
0    6151
1    4825
Name: count, dtype: int64
label
0    56.04
1    43.96
Name: proportion, dtype: float64

TEST
----
label
0    5918
1    4844
Name: count, dtype: int64
label
0    54.99
1    45.01
Name: proportion, dtype: float64


In [14]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

train_df.to_parquet(
    processed_dir / "phishing_train.parquet",
    index=False
)

val_df.to_parquet(
    processed_dir / "phishing_validation.parquet",
    index=False
)

test_df.to_parquet(
    processed_dir / "phishing_test.parquet",
    index=False
)

print("Datasets saved successfully.")

Datasets saved successfully.


In [15]:
X_train = train_df["text"]
y_train = train_df["label"]

X_val = val_df["text"]
y_val = val_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

Training samples: 86933
Validation samples: 10976
Test samples: 10762


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)


X_train_tfidf = tfidf.fit_transform(X_train)

X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)





Train TF-IDF shape: (86933, 96955)
Validation TF-IDF shape: (10976, 96955)
Test TF-IDF shape: (10762, 96955)


In [17]:
from sklearn.linear_model import LogisticRegression

phishing_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

phishing_model.fit(X_train_tfidf, y_train)

print("Phishing model trained successfully.")

Phishing model trained successfully.


In [18]:
val_proba = phishing_model.predict_proba(X_val_tfidf)[:, 1]

print(val_proba[:10])


val_pred = (val_proba >= 0.5).astype(int)

[5.98388166e-04 3.03511834e-02 2.81267471e-04 8.34730441e-02
 9.98734241e-01 9.97795530e-01 9.95538234e-01 9.79485820e-01
 9.98033173e-01 9.77441909e-01]


In [19]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("Precision:", precision_score(y_val, val_pred))
print("Recall:", recall_score(y_val, val_pred))
print("F1:", f1_score(y_val, val_pred))
print("ROC-AUC:", roc_auc_score(y_val, val_proba))
print("PR-AUC:", average_precision_score(y_val, val_proba))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred))

print("\nClassification Report:")
print(classification_report(y_val, val_pred))

Precision: 0.9735482537714404
Recall: 0.9763730569948187
F1: 0.9749586092715232
ROC-AUC: 0.9968345515241213
PR-AUC: 0.9959924579605195

Confusion Matrix:
[[6023  128]
 [ 114 4711]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      6151
           1       0.97      0.98      0.97      4825

    accuracy                           0.98     10976
   macro avg       0.98      0.98      0.98     10976
weighted avg       0.98      0.98      0.98     10976



In [20]:
feature_names = np.array(tfidf.get_feature_names_out())

print("Number of features:", len(feature_names))


coefficients = phishing_model.coef_[0]

print("Number of coefficients:", len(coefficients))


top_phishing_indices = np.argsort(coefficients)[-20:][::-1]

phishing_terms = pd.DataFrame({
    "term": feature_names[top_phishing_indices],
    "coefficient": coefficients[top_phishing_indices]
})

phishing_terms


top_legitimate_indices = np.argsort(coefficients)[:20]

legitimate_terms = pd.DataFrame({
    "term": feature_names[top_legitimate_indices],
    "coefficient": coefficients[top_legitimate_indices]
})

legitimate_terms

Number of features: 96955
Number of coefficients: 96955


,term,coefficient
0,fw,-11.153986
1,re,-9.609713
2,thanks,-8.941148
3,date,-8.394818
4,name,-7.533472
5,the,-6.993091
6,wrote,-6.637102
7,attached,-6.335125
8,list,-6.184696
9,mailing,-5.278993


In [21]:
top_phishing_indices = np.argsort(coefficients)[-20:][::-1]

phishing_terms = pd.DataFrame({
    "term": feature_names[top_phishing_indices],
    "coefficient": coefficients[top_phishing_indices]
})

phishing_terms

,term,coefficient
0,email_separator,13.392225
1,simbol,7.810811
2,your,6.691360
3,quality,5.279460
4,our,5.193466
5,emoji,5.187752
6,reply,5.104105
7,offer,5.032854
8,yourself,4.947808
9,girl,4.742093


In [22]:
tfidf_bigram = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_bigram = tfidf_bigram.fit_transform(X_train)
X_val_bigram = tfidf_bigram.transform(X_val)

print("Train shape:", X_train_bigram.shape)
print("Validation shape:", X_val_bigram.shape)


Train shape: (86933, 882481)
Validation shape: (10976, 882481)


Train the bigram Logistic Regression

In [23]:
phishing_model_bigram = LogisticRegression(
    max_iter=1000,
    random_state=42
)

phishing_model_bigram.fit(X_train_bigram, y_train)

print("Bigram model trained.")


y_val_prob_bigram = phishing_model_bigram.predict_proba(X_val_bigram)[:, 1]

print(y_val_prob_bigram[:10])

Bigram model trained.
[0.00128276 0.05652488 0.0031172  0.17220237 0.9947913  0.99699777
 0.99432981 0.93704778 0.99666329 0.97776647]


In [24]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

y_val_pred_bigram = (y_val_prob_bigram >= 0.5).astype(int)

print("Precision:", precision_score(y_val, y_val_pred_bigram))
print("Recall:", recall_score(y_val, y_val_pred_bigram))
print("F1:", f1_score(y_val, y_val_pred_bigram))
print("ROC-AUC:", roc_auc_score(y_val, y_val_prob_bigram))
print("PR-AUC:", average_precision_score(y_val, y_val_prob_bigram))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_bigram))

Precision: 0.980849292256453
Recall: 0.976580310880829
F1: 0.9787101464326514
ROC-AUC: 0.9975977620219301
PR-AUC: 0.9969977648006816

Confusion Matrix:
[[6059   92]
 [ 113 4712]]


In [25]:
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for threshold in [0.10, 0.20, 0.30, 0.40, 0.50,
                  0.60, 0.70, 0.80, 0.90]:

    y_pred = (y_val_prob_bigram >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_val, y_pred),
        "recall": recall_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred),
        "false_positives": ((y_val == 0) & (y_pred == 1)).sum(),
        "false_negatives": ((y_val == 1) & (y_pred == 0)).sum()
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,threshold,precision,recall,f1,false_positives,false_negatives
0,0.1,0.815794,0.997720,0.897632,1087,11
1,0.2,0.910711,0.995648,0.951287,471,21
2,0.3,0.950169,0.991917,0.970594,251,39
3,0.4,0.966707,0.986943,0.976720,164,63
4,0.5,0.980849,0.976580,0.978710,92,113
5,0.6,0.987681,0.963731,0.975559,58,175
6,0.7,0.992511,0.933886,0.962306,34,319
7,0.8,0.995351,0.887461,0.938315,20,543
8,0.9,0.998083,0.755440,0.859974,7,1180


In [26]:
X_test_bigram = tfidf_bigram.transform(X_test)

y_test_prob_phishing = phishing_model_bigram.predict_proba(
    X_test_bigram
)[:, 1]

y_test_pred_phishing = (
    y_test_prob_phishing >= 0.50
).astype(int)

In [27]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("Precision:",
      precision_score(y_test, y_test_pred_phishing))

print("Recall:",
      recall_score(y_test, y_test_pred_phishing))

print("F1:",
      f1_score(y_test, y_test_pred_phishing))

print("ROC-AUC:",
      roc_auc_score(y_test, y_test_prob_phishing))

print("PR-AUC:",
      average_precision_score(y_test, y_test_prob_phishing))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_phishing))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred_phishing))

Precision: 0.9782653694887187
Recall: 0.9756399669694468
F1: 0.9769509043927649
ROC-AUC: 0.9976959403061214
PR-AUC: 0.9970511130224163

Confusion Matrix:
[[5813  105]
 [ 118 4726]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      5918
           1       0.98      0.98      0.98      4844

    accuracy                           0.98     10762
   macro avg       0.98      0.98      0.98     10762
weighted avg       0.98      0.98      0.98     10762



In [29]:
test_subject = "Finish your application for the AI/ML Developer Internship role at AI Tech Gen Technologies"
test_body = """
Hi Rahul,

It looks like you were in the middle of applying for the AI/ML Developer Internship position. The application is saved, and you can finish it in just a few clicks. Don't let this opportunity pass you by!

Finish My Application
Wishing you the very best for your journey in AI/ML Developer Internship!

Be Unstoppable,
Unstop
"""

test_email = (test_subject.strip() + " " + test_body.strip()).strip()


X_new = tfidf_bigram.transform([test_email])


phishing_probability = phishing_model_bigram.predict_proba(X_new)[0, 1]

phishing_prediction = int(phishing_probability >= 0.50)

print("Phishing probability:", phishing_probability)
print("Prediction:", "PHISHING" if phishing_prediction == 1 else "LEGITIMATE")

Phishing probability: 0.5997972102394094
Prediction: PHISHING


In [30]:
test_subject = "Weekly Statement of Funds & Securities | INDmoney"
test_body = """

Weekly Statement

Hi Rahul Tripathi,

Client Code: NLRV4WRE4Y

We have attached the weekly statement of accounts for funds & securities.

The document is password protected. Please use your PAN in capital letters to access the document.

For any queries, please visit the help section of your INDmoney app

Team INDmoney

Get the INDmoney app!

Available in the App Store or Google Play

"""

test_email = (test_subject.strip() + " " + test_body.strip()).strip()


X_new = tfidf_bigram.transform([test_email])


phishing_probability = phishing_model_bigram.predict_proba(X_new)[0, 1]

phishing_prediction = int(phishing_probability >= 0.50)

print("Phishing probability:", phishing_probability)
print("Prediction:", "PHISHING" if phishing_prediction == 1 else "LEGITIMATE")

Phishing probability: 0.26436884194383725
Prediction: LEGITIMATE
